# Question → JSON plan translator: training on Kaggle

Fine-tunes a small model (LoRA) to turn a short question into a JSON plan, using `train.py` and generated training data.

1. In the right-hand panel: **Session options → Accelerator → GPU T4 x2** (or P100), and **Internet → on**
   (Kaggle asks for a phone-verified account for both).
2. **Save Version → Save & Run All (Commit)**. It runs in the background; the browser can be closed. About 2–2.5 h.
3. When the version has finished, open it → **Output**: `summary.txt` (the dev scores), the `*.predictions.jsonl`
   files (the app scores them) and `qwen06-v6qat.zip` (the model).

In [ ]:
# Settings
MODEL = "Qwen/Qwen3-0.6B"
RUN_NAME = "qwen06-v6qat"
BATCH = 8
QAT = True                   # train against the phone's INT4 rounding (train.py --qat)
MAX_STEPS = -1               # -1: one full pass over the training rows; a number stops earlier

import os
# One GPU, so the run matches the Colab ones (Kaggle's second T4 would double the batch and halve the steps)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
OUT = f"/tmp/runs/{RUN_NAME}"          # checkpoints stay out of the saved output
SAVE = "/kaggle/working"               # only what's worth downloading goes here
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The library versions train.py was tested with
!pip install -q transformers==5.22.0 trl==1.17.0 peft==0.21.2 datasets==5.1.0 accelerate==1.15.0
!pip uninstall -q -y torchao

In [ ]:
# Scripts and data
!rm -rf /tmp/repo && git clone -q --depth 1 https://github.com/fatihsokmen/plan-translator-colab /tmp/repo
!tar xzf /tmp/repo/data.tar.gz -C /tmp/repo && wc -l /tmp/repo/data/*/*.jsonl

In [ ]:
# Training. Prints the dev exact-plan rate at the end.
!cd /tmp/repo && python train.py --model {MODEL} --data data --out {OUT} --bs {BATCH} --save_steps 200 --max_steps {MAX_STEPS} {"--qat" if QAT else ""} 2>&1 | tee -a {OUT}.log
assert os.path.exists(f"{OUT}/final"), "training did not finish: the error is in the output above"

In [ ]:
# The dev score (held-out wordings of this run's data), saved with the run
log = [l.strip() for l in open(f"{OUT}.log")]
new_dev = [l for l in log if l.startswith("dev: exact plan")][-1]
int4_dev = ([l for l in log if l.startswith("dev int4")] or [""])[-1]
open(f"{SAVE}/summary.txt", "w").write(f"{RUN_NAME}\n{new_dev}\n{int4_dev}\n")
print(open(f"{SAVE}/summary.txt").read())

In [ ]:
# Plans for the test and probe sets; the app scores these files (blind_v4 by counts only)
for name, prompts in [("real_like_v2", "data/test/real_like_v2.prompts.jsonl"), ("independent_v3", "data/test/independent_v3.prompts.jsonl"),
                      ("probe_combos_v1", "data/probe/combos_v1.prompts.jsonl"), ("independent_v3_typos", "data/probe/independent_v3_typos.prompts.jsonl"),
                      ("blind_v4", "data/test/blind_v4.prompts.jsonl")]:
    !cd /tmp/repo && python predict.py --model {OUT}/final --prompts {prompts} --out {SAVE}/{name}.predictions.jsonl
!ls -la {SAVE}

In [ ]:
# The model itself, as float16 (about 1.2 GB): finer than bfloat16, so the INT4 rounding the run trained for is kept
import torch, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
m = AutoModelForCausalLM.from_pretrained(f"{OUT}/final", dtype=torch.float16)
m.save_pretrained("/tmp/model_bf16")
AutoTokenizer.from_pretrained(f"{OUT}/final").save_pretrained("/tmp/model_bf16")
shutil.make_archive(f"{SAVE}/{RUN_NAME}", "zip", "/tmp/model_bf16")
!ls -la {SAVE}